# 多クラス・順序分類の許容クラス集合を用いたベイズ最適化

**今回の主役は二値分類ではなく、多クラス分類と順序分類です。** 回帰GPで連続目的値を最大化しながら、分類器が予測する「複数の許容クラスに入る確率」を獲得関数に反映します。

二値分類は既存の特例として扱い、ここでは3クラス以上のモデルを2種類使って、候補提案まで進めます。

[理論](../../docs/theory/30_heterogeneous_classification_constraints.md) / [実装ガイド](../../docs/guide/30_heterogeneous_classification_constraints.md)

## 1. 合成データを用意する

製造条件 `x` に対する連続目的 `score` と、3段階の品質ラベル `quality` を用意します。ラベルは **0=不良、1=要確認、2=良好** と解釈します。許容するのはクラス1と2です。

同じラベルを、順序を仮定しない多クラスGPと、順序を仮定するOrdinal GPでそれぞれ学習し、違いを比較します。

In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.acquisition import make_classification_constrained_qei_acquisition
from robotorchan.models.classification.multiclass.standard.single_task import MulticlassSingleTaskGPClassifier
from robotorchan.models.classification.ordinal.standard.single_task import OrdinalSingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim import optimize_acqf
from robotorchan.semantics import ClassificationConstraint, ProblemSemantics, RegressionObjective

torch.manual_seed(11)
train_X = torch.linspace(0, 1, 15, dtype=torch.double).unsqueeze(-1)
train_Y = 1.0 - (train_X - 0.72).square()
train_quality = torch.bucketize(train_X.squeeze(-1), torch.tensor([0.34, 0.67], dtype=torch.double))
print("各クラスの件数:", torch.bincount(train_quality, minlength=3).tolist())
regression = SingleTaskGP(train_X, train_Y)
multiclass = MulticlassSingleTaskGPClassifier(train_X, train_quality, num_classes=3)
ordinal = OrdinalSingleTaskGPClassifier(train_X, train_quality, num_classes=3)

## 2. 多クラスGPと順序GPを学習する

多クラス分類はクラスを区別しますが、クラス間の順序を必ずしも仮定しません。一方、順序分類では潜在値と閾値を使って順序付きカテゴリの確率を表現します。

いずれも `predict_proba(X)` の最後の次元が3クラスの確率です。ここでは説明用に短めの学習を行います。結果の精度評価を目的とした設定ではありません。

In [ ]:
multiclass.fit(epochs=20, lr=0.03)
ordinal.fit(epochs=20, lr=0.03)
grid_X = torch.linspace(0, 1, 21, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    multi_probs = multiclass.predict_proba(grid_X)
    ordinal_probs = ordinal.predict_proba(grid_X)
print("多クラスの確率形状:", tuple(multi_probs.shape))
print("順序分類の確率形状:", tuple(ordinal_probs.shape))
assert multi_probs.shape == ordinal_probs.shape == (21, 3)

## 3. 複数クラスを許容する確率を比較する

`feasible_classes=(1, 2)` は、要確認または良好の確率を足す指定です。

$$
P(C(x)\\in\\{1,2\\}\\mid D)=P(C(x)=1\\mid D)+P(C(x)=2\\mid D)
$$

順序GPでも許容確率の計算は同じです。ただし、クラス確率自体はモデルの仮定によって変わります。

In [ ]:
allowed_classes = (1, 2)
with torch.no_grad():
    multi_allowed = multi_probs[..., list(allowed_classes)].sum(dim=-1)
    ordinal_allowed = ordinal_probs[..., list(allowed_classes)].sum(dim=-1)
print("x=0.5 付近の許容確率（多クラス）:", multi_allowed[10].item())
print("x=0.5 付近の許容確率（順序）:", ordinal_allowed[10].item())

## 4. モデルごとに回帰qEI×許容確率を構築する

`HeterogeneousModel` で回帰GPと分類GPを組み合わせます。クラス集合の指定は同一ですが、分類モデルが違うため獲得関数の形状や候補は異なり得ます。

この確率重み付けはソフト制約であり、例えば「許容確率80%以上」というハード制約の保証ではありません。

In [ ]:
semantics = ProblemSemantics(
    objectives=(RegressionObjective("score"),),
    constraints=(ClassificationConstraint("quality", feasible_classes=allowed_classes),),
)
bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)
results = {}
for label, classifier in [("multiclass", multiclass), ("ordinal", ordinal)]:
    model = HeterogeneousModel(regression, classifier, output_names=["score", "quality"])
    acquisition = make_classification_constrained_qei_acquisition(
        model,
        semantics,
        best_f=train_Y.max().item(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=11),
    )
    candidate, value = optimize_acqf(
        acquisition, bounds, q=1, optimizer="sobol", raw_samples=128, seed=11
    )
    with torch.no_grad():
        allowed_probability = classifier.predict_proba(candidate)[..., list(allowed_classes)].sum(dim=-1)
    results[label] = (candidate, value, allowed_probability)
    print(label, "候補:", candidate.flatten().tolist(), "獲得値:", value, "許容確率:", allowed_probability)
    assert candidate.shape == (1, 1)

## 5. 結果の読み方と拡張

- 多クラスGPと順序GPの提案候補が一致する必要はありません。学習されたクラス確率とその構造が異なるためです。
- `feasible_classes=(2,)` なら良好のみ、`(1,2)` なら要確認以上を許容する設計です。何を許容するかは業務上の判断です。
- q>1は候補間の許容確率を積で近似します。分類器の候補間相関を考慮した同時確率ではありません。
- `probability_threshold` を付けたハード制約は、この獲得関数ビルダーでは非対応です。
- Mixed変数は対応Optimizerと適切なモデル設計が必要です。今回のNotebookは連続1変数の候補提案を示します。

参考文献と式の前提は[理論章](../../docs/theory/30_heterogeneous_classification_constraints.md)を参照してください。